Seleção dos pacientes por idade e descarga dos registos

In [ ]:
import os

min_age = 40
max_age = 50
PATH_DATASET = f"/media/filipe/Datasets/Datasets/MIMIC-3/MIMIC-3_pacientes_novos_{min_age}_{max_age}"
ADM_PAT_PATH = "/home/filipe/Transferências/mimic-iii-1.4"
os.makedirs(PATH_DATASET, exist_ok=True)


### Primeiro filtrar pela idade e selecionar somente os pacientes que tiveram em UCI por ser mais provável terem dados de ABP

In [ ]:
import pandas as pd
import os

# Carregar ficheiros essenciais
patients = pd.read_csv(f"{ADM_PAT_PATH}/PATIENTS.csv")
admissions = pd.read_csv(f"{ADM_PAT_PATH}/ADMISSIONS.csv")
icustays = pd.read_csv(f"{ADM_PAT_PATH}/ICUSTAYS.csv")

# Converter datas
patients["DOB"] = pd.to_datetime(patients["DOB"], errors="coerce")
admissions["ADMITTIME"] = pd.to_datetime(admissions["ADMITTIME"], errors="coerce")

# Juntar ADMISSIONS + PATIENTS
df = admissions.merge(patients, on="SUBJECT_ID")

# Calcular idade (método oficial)
df["age"] = df["ADMITTIME"].dt.year - df["DOB"].dt.year

# Filtro de idade
df.loc[df["age"] > 89, "age"] = 90

# Filtrar idade desejada
df_age = df[(df["age"] >= min_age) & (df["age"] <= max_age)]


# Filtrar apenas pacientes que passaram pela UCI
icu_subjects = icustays["SUBJECT_ID"].unique()

df_filtrado = df_age[df_age["SUBJECT_ID"].isin(icu_subjects)]
df_filtrado = df_filtrado[df_filtrado["HAS_CHARTEVENTS_DATA"] == 1]

# Lista final de SUBJECT_ID únicos
subject_ids = sorted(df_filtrado["SUBJECT_ID"].unique())

print(f"Pacientes {min_age}–{max_age} anos que passaram pela UCI:")
print(subject_ids[:50])
print("Total:", len(subject_ids))

# Converter para o formato dos ficheiros RECORDS
records_data = [{"subject_id": sid} for sid in subject_ids]

# Criar DataFrame final
df_final = pd.DataFrame(records_data)

# Guardar CSV
output_path = os.path.join(
    PATH_DATASET,
    f"mimic3_pacientes_{min_age}_{max_age}_uci.csv"
)

df_final.to_csv(output_path, index=False)

print(f"\nFicheiro 'mimic3_pacientes_{min_age}_{max_age}_uci.csv' criado.")


### Criar um csv com os urls para fazer o download dos ficheiros numéricos. Assim em vez de fazer o download de todos os ficheiros, baixa-se somente os númericos e analisa-se a presença dos sinais nos mesmos

In [ ]:
import os
import pandas as pd
import requests

# Configurações
INPUT_CSV = f"{PATH_DATASET}/mimic3_pacientes_{min_age}_{max_age}_uci.csv"

df = pd.read_csv(INPUT_CSV)
BASE_URL = "https://physionet.org/files/mimic3wdb-matched/1.0"

# Só se geram as URLs dos ficheiros específicos, que depois se descarregam da PhysioNet (os sinais são de acesso aberto).
records_data = []

print(f"A analisar {len(df)} pacientes para encontrar registos numéricos...")

for sid in df["subject_id"]:
    sid_str = str(sid).zfill(6)
    folder = "p" + sid_str[:2]
    subfolder = "p" + sid_str
    
    # URL do ficheiro RECORDS da pasta do paciente
    # Este ficheiro lista todos os exames reais desse paciente
    records_url = f"{BASE_URL}/{folder}/{subfolder}/RECORDS"
    
    try:
        # O acesso ao RECORDS pode exigir autenticação, consoante o servidor.
        # Com wget/rsync, os nomes dos ficheiros seguem a mesma lógica.
        response = requests.get(records_url)
        if response.status_code == 200:
            # Extrair nomes dos registos
            lines = response.text.strip().split('\n')
            
            # Filtrar apenas os que têm par numérico
            # Nomes que terminam em 'n' (ex: p000109-2141-10-15-10-30n)
            numeric_records = [l for l in lines if l.endswith('n')]
            
            for nr in numeric_records:
                # O registo de sinal é o mesmo nome sem o 'n'
                signal_record = nr[:-1]
                
                records_data.append({
                    "subject_id": sid,
                    "folder_url": f"{BASE_URL}/{folder}/{subfolder}/",
                    "signal_file": signal_record,
                    "numeric_file": nr
                })
            print(f"Paciente {sid_str}: {len(numeric_records)} registos numéricos encontrados.")
        else:
            print(f"Paciente {sid_str}: Ficheiro RECORDS não acessível.")
    except Exception as e:
        print(f"Erro ao aceder {sid_str}: {e}")

# Criar DataFrame com os resultados
df_final = pd.DataFrame(records_data)
output_path = os.path.join(PATH_DATASET, f"lista_para_download_selecionado_{min_age}_{max_age}.csv")
df_final.to_csv(output_path, index=False)

print(f"\nLista de download criada: {output_path}")
print(f"Total de pares (Sinal + Numérico) a baixar: {len(df_final)}")

### Download dos numerics

In [ ]:
import os
import pandas as pd
import requests
from tqdm import tqdm

# --- CONFIGURAÇÃO ---
CSV_LISTA = f"{PATH_DATASET}/lista_para_download_selecionado_{min_age}_{max_age}.csv" # CSV da lista de descarga

def download_file(url, dest_path):
    if os.path.exists(dest_path):
        return
    
    try:
        response = requests.get(url, stream=True, timeout=30)
        if response.status_code == 200:
            os.makedirs(os.path.dirname(dest_path), exist_ok=True)
            with open(dest_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=1024*1024):
                    if chunk:
                        f.write(chunk)
        else:
            # Silencioso para ficheiros que podem não existir (ex: alguns segmentos)
            pass 
    except Exception:
        pass

def iniciar_download():
    if not os.path.exists(CSV_LISTA):
        print(f"Erro: Ficheiro {CSV_LISTA} não encontrado!")
        return

    # Ler o CSV da lista de descarga
    df = pd.read_csv(CSV_LISTA)
    
    tasks = []
    print(f"Mapeando ficheiros para {len(df)} registos...")

    for _, row in df.iterrows():
        # URL base da pasta do paciente
        base_url = row['folder_url']
        if not base_url.endswith('/'):
            base_url += '/'

        # Extrair caminho relativo para manter a estrutura de pastas
        # Tudo o que vem depois de '1.0/'
        rel_path = base_url.split('1.0/')[-1]
        
        # Para cada linha, precisamos de baixar 4 ficheiros:
        # 1. Sinal (.hea e .dat)
        # 2. Numérico (.hea e .dat)
        arquivos = [
            row['signal_file'] + ".hea",
            row['signal_file'] + ".dat",
            row['numeric_file'] + ".hea",
            row['numeric_file'] + ".dat"
        ]
        
        for f in arquivos:
            tasks.append({
                'url': f"{base_url}{f}",
                'dest': os.path.join(PATH_DATASET, "physionet.org/files/mimic3wdb-matched/1.0", rel_path, f)
            })

    print(f"Total de ficheiros a processar: {len(tasks)}")

    # Download com barra de progresso
    for task in tqdm(tasks, desc="Descarregando", unit="arq"):
        download_file(task['url'], task['dest'])

    print(f"\nDownload concluído em: {PATH_DATASET}")

if __name__ == "__main__":
    iniciar_download()

### Verifica a existência de ABP nos numerics

In [ ]:
import os
import pandas as pd

# --- CONFIGURAÇÃO ---
BASE_PATH = f"{PATH_DATASET}"
OUTPUT_CSV = f"{PATH_DATASET}/lista_final_abp_apenas_{min_age}_{max_age}.csv"

def filtrar_apenas_abp():
    valid_records = []
    
    # 1. Procurar todos os n.hea recursivamente
    headers_num = []
    for root, dirs, files in os.walk(BASE_PATH):
        for f in files:
            if f.endswith("n.hea"):
                headers_num.append(os.path.join(root, f))
    
    print(f"Analisando {len(headers_num)} ficheiros numéricos...")

    for path in headers_num:
        try:
            with open(path, 'r') as f_obj:
                content = f_obj.read().upper()
                
                # CRITÉRIO RÍGIDO: Tem de conter explicitamente 'ABP'
                # 'NBP' (não invasiva) fica de fora
                if 'ABP' in content:
                    parts = path.split(os.sep)
                    # Encontrar pXX (ex: p01) e pXXXXXX (ex: p019412)
                    p_folder = next(p for p in parts if p.startswith('p') and len(p) == 3)
                    p_id_folder = next(p for p in parts if p.startswith('p') and len(p) == 7)
                    
                    base_url = f"https://physionet.org/files/mimic3wdb-matched/1.0/{p_folder}/{p_id_folder}/"
                    
                    nome_n_hea = os.path.basename(path).replace('.hea', '')
                    nome_sig = nome_n_hea[:-1] 
                    
                    valid_records.append({
                        'subject_id': p_id_folder.replace('p', ''),
                        'folder_url': base_url,
                        'signal_file': nome_sig,
                        'numeric_file': nome_n_hea
                    })
        except Exception as e:
            print(f"Erro ao ler {path}: {e}")

    if valid_records:
        df = pd.DataFrame(valid_records)
        df.to_csv(OUTPUT_CSV, index=False)
        print(f"\nFILTRAGEM ABP CONCLUÍDA!")
        print(f"Dos {len(headers_num)} ficheiros, {len(df)} têm Pressão Arterial Invasiva (ABP).")
        print(f"Lista guardada em: {OUTPUT_CSV}")
    else:
        print("Nenhum registo com ABP encontrado. Talvez estes pacientes só tenham NBP.")

if __name__ == "__main__":
    filtrar_apenas_abp()

### Baixa todos os headers dos pacientes que tem registo de ABP nos ficheiros numéricos

In [ ]:
import os
import pandas as pd
import requests
from tqdm import tqdm
import re

# --- CONFIGURAÇÃO ---
CSV_INPUT = f"{PATH_DATASET}/lista_final_abp_apenas_{min_age}_{max_age}.csv"
BASE_OUTPUT = f"{PATH_DATASET}"

def get_all_hea_links(url):
    """Encontra todos os ficheiros .hea na página da pasta do paciente."""
    try:
        response = requests.get(url, timeout=15)
        if response.status_code == 200:
            # Procura por ficheiros que terminam em .hea
            return re.findall(r'href="([^"]+\.hea)"', response.text)
    except:
        return []
    return []

def baixar_headers_noite():
    if not os.path.exists(CSV_INPUT):
        print(f"Erro: Ficheiro {CSV_INPUT} não encontrado.")
        return

    df = pd.read_csv(CSV_INPUT)
    # URLs únicas das pastas dos pacientes
    pastas_unicas = df['folder_url'].unique()
    
    print(f"Iniciando download de cabeçalhos para {len(pastas_unicas)} pacientes...")
    
    total_baixados = 0

    for url_base in tqdm(pastas_unicas, desc="Pastas"):
        # Criar estrutura de pastas local
        rel_path = url_base.split('1.0/')[-1]
        local_dir = os.path.join(BASE_OUTPUT, "physionet.org/files/mimic3wdb-matched/1.0", rel_path)
        os.makedirs(local_dir, exist_ok=True)
        
        # Obter lista de todos os .hea naquela pasta
        headers = get_all_hea_links(url_base)
        
        for hea in headers:
            local_path = os.path.join(local_dir, hea)
            
            # Só baixa se não existir
            if not os.path.exists(local_path):
                try:
                    r = requests.get(url_base + hea, timeout=1)
                    if r.status_code == 200:
                        with open(local_path, 'w') as f:
                            f.write(r.text)
                        total_baixados += 1
                except:
                    continue

    print(f"\nConcluído! Foram baixados {total_baixados} ficheiros .hea.")
    print(f"Estão todos organizados em: {BASE_OUTPUT}")

if __name__ == "__main__":
    baixar_headers_noite()

### Analisa todos os headers e entrega um csv com todos os records com os três sinais

In [ ]:
import os
import pandas as pd
from tqdm import tqdm

# --- CONFIGURAÇÃO ---
BASE_PATH = f"{PATH_DATASET}/physionet.org/files/mimic3wdb-matched/1.0"
CSV_INPUT = f"{PATH_DATASET}/lista_final_abp_apenas_{min_age}_{max_age}.csv"
CSV_OUTPUT = f"{PATH_DATASET}/lista_download_sinais_final_{min_age}_{max_age}.csv"

def analisar_sinais_locais():
    df_records = pd.read_csv(CSV_INPUT)
    resultados = []
    
    print("Analisando cabeçalhos para identificar segmentos com ECG + PLETH + ABP...")

    # Percorrer cada record que filtramos anteriormente
    for _, row in tqdm(df_records.iterrows(), total=len(df_records)):
        subject_id = row['subject_id']
        folder_url = row['folder_url']
        sig_master = row['signal_file']
        
        # Caminho local da pasta do paciente
        # Ex: .../p00/p000652/
        rel_path = folder_url.split('1.0/')[-1]
        local_dir = os.path.join(BASE_PATH, rel_path)
        
        master_hea_path = os.path.join(local_dir, f"{sig_master}.hea")
        
        if not os.path.exists(master_hea_path):
            continue

        with open(master_hea_path, 'r') as f:
            linhas = f.readlines()
            
        for linha in linhas:
            parts = linha.split()
            if len(parts) >= 2 and not parts[0].startswith('#'):
                seg_name = parts[0]
                # Ignorar placeholders (~ ou o próprio nome do master)
                if seg_name == '~' or seg_name == sig_master:
                    continue
                
                seg_hea_path = os.path.join(local_dir, f"{seg_name}.hea")
                
                if os.path.exists(seg_hea_path):
                    with open(seg_hea_path, 'r') as f_seg:
                        conteudo = f_seg.read().upper()
                        
                        # Critério: os 3 sinais síncronos
                        tem_abp = 'ABP' in conteudo
                        tem_pleth = 'PLETH' in conteudo
                        tem_ecg = any(x in conteudo for x in ['II', 'V', 'ECG', 'MLII'])
                        
                        if tem_abp and tem_pleth and tem_ecg:
                            # Se o segmento é válido, guardamos o URL do .dat correspondente
                            resultados.append({
                                'subject_id': subject_id,
                                'record_master': sig_master,
                                'segment_name': seg_name,
                                'dat_url': f"{folder_url}{seg_name}.dat",
                                'local_path': os.path.join(local_dir, f"{seg_name}.dat")
                            })

    if resultados:
        df_final = pd.DataFrame(resultados)
        df_final.to_csv(CSV_OUTPUT, index=False)
        print(f"\nANÁLISE CONCLUÍDA!")
        print(f"Encontrados {len(df_final)} segmentos válidos com os 3 sinais.")
        print(f"Lista para download guardada em: {CSV_OUTPUT}")
    else:
        print("\nNenhum segmento encontrado com os 3 sinais simultâneos.")

if __name__ == "__main__":
    analisar_sinais_locais()

### Download de 5 segmentos por paciente

In [ ]:
import pandas as pd
import os
import subprocess

# 1. Configurações
csv_path = f"{PATH_DATASET}/lista_download_sinais_final_{min_age}_{max_age}.csv"
df = pd.read_csv(csv_path)

# Filtrar apenas dados reais (ignorar ficheiros de layout)
df_sinais = df[~df['segment_name'].str.contains('layout')].copy()

print(f"Motor de Download: WGET")
print(f"Alvo: 5 ficheiros bem-sucedidos por paciente.\n")

for subject, group in df_sinais.groupby('subject_id'):
    print(f"Paciente {subject}:")
    
    ok_count = 0
    
    for idx, row in group.iterrows():
        # Para assim que atingir a meta de 5
        if ok_count >= 5:
            break
            
        url = row['dat_url']
        local_path = row['local_path']
        filename = row['segment_name'] + ".dat"
        
        # Garante que a pasta de destino existe
        os.makedirs(os.path.dirname(local_path), exist_ok=True)

        # Se já existir, contamos como sucesso e avançamos
        if os.path.exists(local_path) and os.path.getsize(local_path) > 0:
            print(f"   [{ok_count+1}/5] {filename} já existe localmente.")
            ok_count += 1
            continue
            
        try:
            print(f"   [{ok_count+1}/5] Wget a baixar {filename}...", end=" ", flush=True)
            
            # comando wget otimizado:
            # -q: modo silencioso
            # -c: continua se o download foi interrompido
            # --timeout: desiste se o servidor não responder em X segundos
            # --tries: tenta apenas 2 vezes antes de dar erro
            # -O: grava no caminho exato de local_path
            cmd = [
                "wget", "-q", "-c", 
                url, 
                "-O", local_path
            ]
            
            # Executa o comando no sistema
            result = subprocess.run(cmd, capture_output=True)
            
            if result.returncode == 0:
                print("OK!")
                ok_count += 1
            else:
                print("FALHOU (A saltar para o próximo...)")
                # Remove se o ficheiro ficou com 0 bytes ou erro
                if os.path.exists(local_path) and os.path.getsize(local_path) == 0:
                    os.remove(local_path)
                    
        except Exception as e:
            print(f"ERRO: {e}")

print("\nProcesso concluído com o motor Wget!")

### Depois de todos os records estarem baixados verifica a existência dos sinais em cada record

In [ ]:
import os
import pandas as pd
import wfdb
import numpy as np
from tqdm import tqdm

# --- CONFIGURAÇÃO ---
BASE_PATH = f"{PATH_DATASET}/physionet.org/files/mimic3wdb-matched/1.0"
   
PATIENTS_CSV_PATH = f"{ADM_PAT_PATH}/PATIENTS.csv"
ADMISSIONS_CSV_PATH = f"{ADM_PAT_PATH}/ADMISSIONS.csv"
OUTPUT_CSV = f"{PATH_DATASET}/dataset_final_para_treino_{min_age}_{max_age}.csv"
MIN_DURATION_SEC = 10 

def varrimento_e_validacao():
    # 1. Carregar dados dos pacientes (Género)
# 1. Carregar dados
    print(f"A carregar informações dos pacientes...")
    df_p = pd.read_csv(PATIENTS_CSV_PATH)
    df_a = pd.read_csv(ADMISSIONS_CSV_PATH)

    # 2. Converter datas com segurança (errors='coerce' evita crash com datas inválidas)
    df_p["DOB"] = pd.to_datetime(df_p["DOB"], errors="coerce")
    df_a["ADMITTIME"] = pd.to_datetime(df_a["ADMITTIME"], errors="coerce")

    # 3. Merge inicial para ter as datas de nascimento e admissão juntas
    df_merged = df_a.merge(df_p[['SUBJECT_ID', 'DOB', 'GENDER']], on="SUBJECT_ID")

    # 4. Calcular idade (pelo ano para evitar Overflow) e tratar anonimização (>89 -> 90)
    df_merged['age'] = df_merged['ADMITTIME'].dt.year - df_merged['DOB'].dt.year
    df_merged.loc[df_merged['age'] > 89, 'age'] = 90

    # 5. RESOLUÇÃO DOS DUPLICADOS:
    # Ordenamos por data de admissão e pegamos na PRIMEIRA vez que o paciente aparece.
    # Isto garante que o SUBJECT_ID seja ÚNICO.
    df_patients_unique = df_merged.sort_values('ADMITTIME').groupby('SUBJECT_ID').agg({
        'age': 'first',
        'GENDER': 'first'
    }).reset_index()

    # 6. Converter ID para string para o merge posterior com os ficheiros de sinais
    df_patients_unique['SUBJECT_ID'] = df_patients_unique['SUBJECT_ID'].astype(str)
    
    # Substituímos a variável df_patients pela versão limpa
    df_patients = df_patients_unique

    records_validados = []
    print(f"Iniciando varrimento em: {BASE_PATH}")
    
    all_headers = []
    for root, dirs, files in os.walk(BASE_PATH):
        for f in files:
            if f.endswith('.hea') and not f.endswith('n.hea') and '_' in f:
                all_headers.append(os.path.join(root, f))

    print(f"Encontrados {len(all_headers)} candidatos. Validando sinais...")

    for hea_path in tqdm(all_headers):
        record_path = hea_path.replace('.hea', '')
        
        if not os.path.exists(record_path + '.dat'):
            continue

        try:
            header = wfdb.rdheader(record_path)
            
            duration_sec = header.sig_len / header.fs
            if duration_sec < MIN_DURATION_SEC:
                continue
            
            sig_names = [s.upper() for s in header.sig_name]
            tem_abp = 'ABP' in sig_names
            tem_pleth = 'PLETH' in sig_names or 'PPG' in sig_names
            tem_ecg = any(x in sig_names for x in ['II', 'V', 'ECG', 'MLII', 'I'])
            
            if tem_abp and tem_pleth and tem_ecg:
                mid_point = int(header.sig_len / 2)
                sample_size = min(int(header.fs * 5), header.sig_len)
                
                record = wfdb.rdrecord(record_path, sampfrom=mid_point, sampto=mid_point + sample_size)
                
                if np.any(np.std(record.p_signal, axis=0) == 0):
                    continue

                # Extrair o ID do paciente da pasta (ex: p012345 -> 12345)
                parts = record_path.split(os.sep)
                raw_subject_id = parts[-2] # p012345
                # Limpar o 'p' e zeros à esquerda para bater com o CSV dos pacientes
                clean_id = raw_subject_id.replace('p', '').lstrip('0')

                records_validados.append({
                    'subject_id_clean': clean_id, # ID para o merge
                    'subject_folder': raw_subject_id, # ID original da pasta
                    'segment': os.path.basename(record_path),
                    'duration_min': round(duration_sec / 60, 2),
                    'fs': header.fs,
                    'channels': ", ".join(header.sig_name),
                    'absolute_path': record_path
                })

        except Exception:
            continue

    # 2. Criar DataFrame e fazer o MERGE com o Género
    if records_validados:
        df_waves = pd.DataFrame(records_validados)
        
        # Merge final
        df_final = pd.merge(df_waves, df_patients, left_on='subject_id_clean', right_on='SUBJECT_ID', how='left')
        
        # Remover a coluna de ID repetida e renomear
        df_final = df_final.drop(columns=['SUBJECT_ID'])
        df_final = df_final.rename(columns={'GENDER': 'gender'})

        df_final.to_csv(OUTPUT_CSV, index=False)
        
        print(f"\nSUCESSO!")
        print(f"{len(df_final)} segmentos validados com informação de género.")
        print(f"Distribuição por género:\n{df_final['gender'].value_counts()}")
        print(f"Lista guardada em: {OUTPUT_CSV}")
    else:
        print("\nNenhum ficheiro válido encontrado.")

if __name__ == "__main__":
    varrimento_e_validacao()

### Copiar vários pacientes para uma pasta única

In [ ]:
import os
import pandas as pd
import shutil
from tqdm import tqdm
import glob

# --- CONFIGURAÇÃO ---
PASTAS_RAIZ = [
    # "/media/filipe/Dados/MIMIC-4",,
    "/media/filipe/Datasets/Datasets/MIMIC-3/MIMIC-3_pacientes_novos_40_50",
    # "/media/filipe/Datasets/Datasets/MIMIC-3/MIMIC-3_pacientes_novos_26_39",
]

# Pasta onde ficam centralizadas as pastas dos pacientes
PASTA_DESTINO_FINAL = "/media/filipe/Dados/MIMIC_DATASET_CONSOLIDADO"

def consolidar_por_csv_index():
    if not os.path.exists(PASTA_DESTINO_FINAL):
        os.makedirs(PASTA_DESTINO_FINAL)

    pacientes_copiados = set()

    for pasta_base in PASTAS_RAIZ:
        if not os.path.exists(pasta_base):
            print(f"Pasta raiz não encontrada: {pasta_base}")
            continue

        # CSVs de indexação (MIMIC-3 e MIMIC-4)
        # O padrão busca ficheiros que comecem por 'dataset_final_para_treino'
        ficheiros_index = glob.glob(os.path.join(pasta_base, "dataset_final_para_treino_*.csv"))

        for csv_path in ficheiros_index:
            print(f"\nLendo index: {os.path.basename(csv_path)}")
            try:
                df = pd.read_csv(csv_path)
                
                # Verificação de colunas críticas
                if 'absolute_path' not in df.columns or 'subject_folder' not in df.columns:
                    print(f"  Colunas necessárias não encontradas em {os.path.basename(csv_path)}")
                    continue

                # Itera pelos registros do CSV
                for _, row in tqdm(df.iterrows(), total=len(df), desc="Pacientes"):
                    sub_id = str(row['subject_folder'])
                    abs_path = row['absolute_path']
                    
                    # Se este paciente já foi consolidado, ignoramos
                    if sub_id in pacientes_copiados:
                        continue

                    # O absolute_path aponta para o ficheiro (ex: .../p000188/3317157_0002)
                    # O dirname dá-nos a pasta do paciente (ex: .../p000188)
                    pasta_origem_paciente = os.path.dirname(abs_path)
                    pasta_destino_paciente = os.path.join(PASTA_DESTINO_FINAL, sub_id)

                    if os.path.exists(pasta_origem_paciente):
                        if not os.path.exists(pasta_destino_paciente):
                            try:
                                # Copia a pasta inteira do paciente com todos os sinais
                                os.system(f"rsync -ah --progress '{pasta_origem_paciente}/' '{pasta_destino_paciente}'")
                                pacientes_copiados.add(sub_id)
                            except Exception as e:
                                print(f"\n  Erro ao copiar {sub_id}: {e}")
                        else:
                            # Se já existe no destino (talvez de outra pasta raiz), apenas registamos
                            pacientes_copiados.add(sub_id)
                    else:
                        # Caso o caminho no CSV esteja quebrado ou o disco não esteja montado
                        pass 

            except Exception as e:
                print(f"  Erro ao processar CSV {csv_path}: {e}")

    print("\n" + "="*45)
    print("CONSOLIDAÇÃO CONCLUÍDA COM SUCESSO")
    print(f"Destino: {PASTA_DESTINO_FINAL}")
    print(f"Total de pacientes únicos movidos: {len(pacientes_copiados)}")
    print("="*45)

if __name__ == "__main__":
    consolidar_por_csv_index()

### Eliminar ficheiros não utilizados

In [ ]:
import os

# ==============================================================================
# CONFIGURAÇÕES COHORT & DIRETORIA
# ==============================================================================
# 1. Caminho para a pasta onde estão os ficheiros originais
PASTA_DATASET = "/media/filipe/Dados/MIMIC_DATASET_CONSOLIDADO"

# 2. O ficheiro âncora gerado automaticamente pela sua função de carregamento
FICHEIRO_ANCORA = "../../03_ABSOLUTO_codigo_e_resultados/RESULTADOS/pacientes_validos_cohort.txt"

# 3. MODO DE SEGURANÇA (DRY RUN)
# True  = Apenas simula e mostra o que seria apagado (NÃO APAGA NADA).
# False = APAGA OS FICHEIROS DEFINITIVAMENTE DO DISCO.
DRY_RUN = False
# ==============================================================================

# Verificar se o ficheiro de âncora existe
if not os.path.exists(FICHEIRO_ANCORA):
    raise FileNotFoundError(
        f"O ficheiro '{FICHEIRO_ANCORA}' não foi encontrado!\n"
        f"Execute primeiro a sua função 'carregar_e_processar_todos_deltas'."
    )

# Carregar os IDs dos 357 pacientes da coorte para um Set
with open(FICHEIRO_ANCORA, "r") as f:
    pacientes_validos = set(linha.strip() for linha in f if linha.strip())

print(f"Âncora carregada: {len(pacientes_validos)} pacientes válidos identificados.")
print(f"Modo DRY_RUN: {'ATIVADO (Simulação Segura)' if DRY_RUN else 'DESATIVADO (AVISO: VAI APAGAR!)'}\n")

# Contadores para o relatório final
contadores = {"mantidos": 0, "eliminados": 0, "bytes_poupados": 0}

# Percorrer recursivamente todas as subpastas e ficheiros do dataset
for raiz, diretorias, ficheiros in os.walk(PASTA_DATASET):
    
    # PASSO CHAVE: Analisar o caminho da pasta atual (raiz)
    # Exemplo: "/.../MIMIC_DATASET_CONSOLIDADO/p040867" -> extrai "p040867"
    partes_do_caminho = os.path.normpath(raiz).split(os.sep)
    
    id_paciente_pasta = None
    for parte in partes_do_caminho:
        # O padrão do MIMIC começa por 'p' seguido de dígitos (ex: p040867)
        if parte.startswith('p') and parte[1:].isdigit():
            id_paciente_pasta = parte
            break

    for ficheiro in ficheiros:
        caminho_completo = os.path.join(raiz, ficheiro)
        
        # ESCUDO 1: Nunca, em caso algum, apagar CSVs, Scripts ou Notebooks
        if ficheiro.endswith(('.csv', '.py', '.ipynb', '.txt', '.json', '.md')):
            continue  # Salta o ficheiro imediatamente de forma segura
            
        # ESCUDO 2: Só avaliar ficheiros de sinal/cabeçalho reais do MIMIC
        if ficheiro.endswith(('.hea', '.mat', '.dat')):
            
            # Se encontramos uma pasta de paciente válida no caminho atual
            if id_paciente_pasta is not None:
                if id_paciente_pasta in pacientes_validos:
                    # O paciente da pasta é válido, o ficheiro não se toca!
                    contadores["mantidos"] += 1
                else:
                    # A pasta pertence a um paciente REJEITADO nos filtros clínicos!
                    try:
                        tamanho_bytes = os.path.getsize(caminho_completo)
                        contadores["bytes_poupados"] += tamanho_bytes
                        contadores["eliminados"] += 1
                        
                        if DRY_RUN:
                            print(f"[SIMULAÇÃO] Iria eliminar: {ficheiro} (Pasta do Paciente: {id_paciente_pasta})")
                        else:
                            os.remove(caminho_completo)
                            print(f"Eliminado: {ficheiro} (Paciente: {id_paciente_pasta})")
                    except Exception as e:
                        print(f"Erro ao processar/eliminar {ficheiro}: {e}")
            else:
                # Ficheiro de sinal perdido fora de uma pasta de doente (Manter por segurança)
                contadores["mantidos"] += 1

# --- RELATÓRIO FINAL DE PERFORMANCE EM DISCO ---
espaco_gb = contadores["bytes_poupados"] / (1024 * 1024 * 1024)
print("\n" + "="*60)
print("RELATÓRIO FINAL DE LIMPEZA CLÍNICA BRINDADA")
print("="*60)
print(f"Ficheiros de sinal mantidos (Dos 357 doentes): {contadores['mantidos']}")
print(f"Ficheiros de sinal a eliminar (Pacientes rejeitados): {contadores['eliminados']}")
print(f"Espaço total em disco a libertar: {espaco_gb:.2f} GB")

if DRY_RUN:
    if contadores["eliminados"] > 0:
        print("\nVerificação concluída com sucesso!")
        print("Se a lista de ficheiros acima parecer correta, mude para:")
        print("DRY_RUN = False")
        print("E corra o script novamente para limpar o disco de forma definitiva.")
    else:
        print("\nO seu dataset já está perfeitamente limpo! Nenhum ficheiro extra foi encontrado.")
print("="*60)

### Baixar segmentos adjacentes

In [ ]:
import os
import time
import urllib.request
from urllib.error import HTTPError

# ==============================================================================
# CONFIGURAÇÕES COHORT & PHYSIONET
# ==============================================================================
# 1. Caminho absoluto da pasta consolidada
PASTA_DATASET = "/media/filipe/Dados/MIMIC_DATASET_CONSOLIDADO"

# 2. Ficheiro âncora dos 357 pacientes da coorte
FICHEIRO_ANCORA = "pacientes_teste.txt"

# 3. URL Base do MIMIC-III Waveform Database Matched no PhysioNet
BASE_URL = "https://physionet.org/files/mimic3wdb-matched/1.0/"

# 4. Intervalo de cortesia entre downloads (em segundos) para evitar bloqueios
DELAY_ENTRE_DOWNLOADS = 0.1
# ==============================================================================

def descarregar_url(url, destino_local):
    """Função auxiliar para descarregar um ficheiro com User-Agent simulado."""
    try:
        # O PhysioNet às vezes bloqueia scripts que não se identificam como um browser
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as resposta, open(destino_local, 'wb') as f_saida:
            f_saida.write(resposta.read())
        return True
    except HTTPError as e:
        if e.code == 404:
            return False # Ficheiro não existe no servidor (esperado para alguns vizinhos)
        print(f"\nErro HTTP {e.code} ao tentar aceder a: {url}")
        return False
    except Exception as e:
        print(f"\nErro inesperado no download: {e}")
        return False

# 1. Validar e carregar a Âncora
if not os.path.exists(FICHEIRO_ANCORA):
    raise FileNotFoundError(f"O ficheiro '{FICHEIRO_ANCORA}' não foi encontrado.")

with open(FICHEIRO_ANCORA, "r") as f:
    pacientes_validos = set(linha.strip() for filename in f for linha in [filename] if linha.strip())

print(f"Âncora carregada: Focado estritamente em {len(pacientes_validos)} pacientes.")

# 2. Mapear os segmentos que já estão no disco (lógica segura de pastas)
# Estrutura: { record_id: { 'patient_id': 'pXXXXXX', 'segs': set([1, 2, ...]), 'raiz': 'caminho' } }
segmentos_locais = {}
print("A analisar a estrutura de pastas local para mapear segmentos existentes...")

for raiz, diretorias, ficheiros in os.walk(PASTA_DATASET):
    partes_do_caminho = os.path.normpath(raiz).split(os.sep)
    
    id_paciente_pasta = None
    for parte in partes_do_caminho:
        if parte in pacientes_validos:
            id_paciente_pasta = parte
            break
            
    if not id_paciente_pasta:
        continue
        
    for f in ficheiros:
        if f.endswith(('.hea', '.mat', '.dat')) and '_' in f:
            nome_sem_ext = f.rsplit('.', 1)[0]
            try:
                record_id, seg_idx_str = nome_sem_ext.rsplit('_', 1)
                if seg_idx_str.isdigit():
                    seg_idx = int(seg_idx_str)
                    
                    if record_id not in segmentos_locais:
                        segmentos_locais[record_id] = {
                            'patient_id': id_paciente_pasta,
                            'raiz_local': raiz,
                            'segs': set()
                        }
                    segmentos_locais[record_id]['segs'].add(seg_idx)
            except ValueError:
                continue

print(f"Mapeamento concluído. Encontradas {len(segmentos_locais)} sessões (record_ids) ativas.")

# 3. Calcular vizinhos em falta e iniciar o download dinâmico
print("\nA iniciar a busca e download de vizinhos adjacentes no PhysioNet...")
total_baixado = 0
total_404 = 0

for record_id, dados in segmentos_locais.items():
    patient_id = dados['patient_id']
    segs_presentes = dados['segs']
    pasta_destino = dados['raiz_local']
    subpasta_prefixo = patient_id[:3] # Ex: 'p04'
    
    # Para cada segmento existente, verificar o anterior (N-1) e o seguinte (N+1)
    vizinhos_alvo = set()
    for seg in segs_presentes:
        for adj in [seg - 5, seg - 4, seg -3, seg - 2, seg - 1, seg + 1, seg + 2, seg + 3, seg + 4, seg + 5]: # Verificar até 5 segmentos adjacentes para aumentar as chances
            if adj > 0 and adj not in segs_presentes:
                vizinhos_alvo.add(adj)
                
    if not vizinhos_alvo:
        continue
        
    for adj_seg in sorted(vizinhos_alvo):
        adj_seg_str = f"{adj_seg:04d}"
        nome_segmento_adj = f"{record_id}_{adj_seg_str}"
        
        # Construir o URL base remoto e o caminho local para este segmento específico
        url_remoto_base = f"{BASE_URL}{subpasta_prefixo}/{patient_id}/{nome_segmento_adj}"
        caminho_local_base = os.path.join(pasta_destino, nome_segmento_adj)
        
        # --- PASSO INTELIGENTE: Testar primeiro o .hea ---
        url_hea = f"{url_remoto_base}.hea"
        local_hea = f"{caminho_local_base}.hea"
        
        # Se o ficheiro .hea local já existir por algum motivo, assumimos que está feito
        if os.path.exists(local_hea):
            continue
            
        print(f"⏳ A testar segmento adjacente: {nome_segmento_adj}...", end="", flush=True)
        
        # Tenta o download do cabeçalho
        sucesso_hea = descarregar_url(url_hea, local_hea)
        
        if sucesso_hea:
            print(" [CABEÇALHO OK] -> A baixar sinais...", end="", flush=True)
            total_baixado += 1
            
            # Como o cabeçalho existe, descarrega obrigatoriamente os ficheiros de sinal (.mat e .dat)
            for ext in ['.mat', '.dat']:
                url_sinal = f"{url_remoto_base}{ext}"
                local_sinal = f"{caminho_local_base}{ext}"
                
                descarregar_url(url_sinal, local_sinal)
                time.sleep(DELAY_ENTRE_DOWNLOADS)
                
            print(" Concluído!")
        else:
            # Se deu 404 no .hea, o segmento simplesmente não existe no PhysioNet
            total_404 += 1
            print(" Não existe no PhysioNet (Ignorado).")
            
        time.sleep(DELAY_ENTRE_DOWNLOADS)

print("\n" + "="*60)
print("RELATÓRIO FINAL DE EXPANSÃO COHORT")
print("="*60)
print(f"Novos segmentos adjacentes integrados com sucesso: {total_baixado}")
print(f"Combinações teóricas testadas que não existem no servidor: {total_404}")
print(f"Todos os ficheiros foram guardados nas respetivas pastas em:\n   -> {PASTA_DATASET}")
print("="*60)

In [ ]:
import os
import time
import urllib.request
import pandas as pd
from urllib.error import HTTPError

# ==============================================================================
# CONFIGURAÇÕES DE DIRETORIA, CSVs E LIMIAR
# ==============================================================================
PASTA_DATASET = "/media/filipe/Dados/MIMIC_DATASET_CONSOLIDADO"
FICHEIRO_ANCORA = "../../03_ABSOLUTO_codigo_e_resultados/RESULTADOS/pacientes_validos_cohort.txt"

# Caminhos dos dois CSVs de características
FICHEIROS_CSV = [

    "/media/filipe/Dados/MIMIC_DATASET_CONSOLIDADO/features_mulheres_teste_novo2.csv",

    "/media/filipe/Dados/MIMIC_DATASET_CONSOLIDADO/features_homens_teste_novo2.csv"

]

# LIMIAR CRÍTICO: Só vai auditar e baixar para quem tiver MENOS janelas que isto
LIMIAR_JANELAS = 100 

# Configurações do PhysioNet
BASE_URL = "https://physionet.org/files/mimic3wdb-matched/1.0/"
DELAY_ENTRE_DOWNLOADS = 0.1
# ==============================================================================

def descarregar_url(url, destino_local):
    """Função auxiliar para descarregar ficheiros simulando um browser."""
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as resposta, open(destino_local, 'wb') as f_saida:
            f_saida.write(resposta.read())
        return True
    except HTTPError as e:
        if e.code == 404:
            return False
        print(f"\nErro HTTP {e.code} em: {url}")
        return False
    except Exception as e:
        print(f"\nErro inesperado: {e}")
        return False

# --- PASSO 1: Carregar os Doentes Válidos (Âncora do TXT) ---
if not os.path.exists(FICHEIRO_ANCORA):
    raise FileNotFoundError(f"Ficheiro mestre '{FICHEIRO_ANCORA}' não encontrado!")

with open(FICHEIRO_ANCORA, "r") as f:
    # Garante que guarda o ID exatamente como está no TXT (ex: 'p10126957')
    pacientes_validos = set(linha.strip() for linha in f if linha.strip())

print(f"Ficheiro Âncora lido: {len(pacientes_validos)} pacientes mapeados no TXT.")

# --- PASSO 2: Carregar e Normalizar os CSVs (Removendo o 'p' se existir) ---
print("A contabilizar janelas globais dos CSVs (Mulheres + Homens)...")
contagens_lista = []

for csv_path in FICHEIROS_CSV:
    if os.path.exists(csv_path):
        df_tmp = pd.read_csv(csv_path)
        if 'subject_id' in df_tmp.columns:
            # CORREÇÃO CRUCIAL: Forçar o ID a ser string e remover qualquer 'p' para uniformizar
            df_tmp['subject_id'] = df_tmp['subject_id'].astype(str).str.replace('p', '', case=False)
            contagens_lista.append(df_tmp['subject_id'].value_counts())

# Combinar as contagens de todos os CSVs
if contagens_lista:
    contagem_janelas = pd.concat(contagens_lista).groupby(level=0).sum()
else:
    contagem_janelas = pd.Series()

# --- PASSO 3: Filtrar apenas os doentes críticos dentro da coorte ---
pacientes_criticos = set()
for p in pacientes_validos:
    # Removemos o 'p' do ID do TXT apenas para conseguir pesquisar dentro do CSV
    p_limpo = p.replace('p', '')
    
    total_janelas_no_csv = contagem_janelas.get(p_limpo, 0)
    
    if total_janelas_no_csv < LIMIAR_JANELAS:
        # Guardamos com o 'p' para conseguir fazer os matches das pastas físicas no disco
        pacientes_criticos.add(p)

print(f"Filtragem Concluída: Apenas {len(pacientes_criticos)} dos {len(pacientes_validos)} pacientes estão abaixo de {LIMIAR_JANELAS} janelas.")

if not pacientes_criticos:
    print("Incrível! Todos os pacientes do TXT já têm dados suficientes. Processo interrompido.")
    exit()

# --- PASSO 4: Varrer as pastas locais só dos doentes críticos filtrados ---
mapeamento_criticos = {}
print("A varrer o disco à procura dos registos mestre destes pacientes necessitados...")

for raiz, diretorias, ficheiros in os.walk(PASTA_DATASET):
    partes = os.path.normpath(raiz).split(os.sep)
    id_paciente = next((p for p in partes if p in pacientes_criticos), None)
    
    if not id_paciente:
        continue
        
    if id_paciente not in mapeamento_criticos:
        mapeamento_criticos[id_paciente] = {'raiz': raiz, 'masters': set()}
        
    for f in ficheiros:
        if f.endswith('.hea') and '_' not in f:
            master_id = f.replace('n.hea', '').replace('.hea', '')
            if master_id.isdigit():
                mapeamento_criticos[id_paciente]['masters'].add(master_id)

# --- PASSO 5: Processar a Auditoria de Sinais de Alta Frequência ---
print("\nA iniciar a auditoria de sinais triplos e download focado...")
total_novos_segmentos = 0

for patient_id, dados in mapeamento_criticos.items():
    local_dir = dados['raiz']
    masters = dados['masters']
    subpasta_p = patient_id[:3]
    
    # Buscar a contagem real usando o ID limpo
    janelas_atuais = contagem_janelas.get(patient_id.replace('p', ''), 0)
    
    for sig_master in sorted(masters):
        folder_url = f"{BASE_URL}{subpasta_p}/{patient_id}/"
        master_hea_local = os.path.join(local_dir, f"{sig_master}.hea")
        
        if not os.path.exists(master_hea_local):
            print(f"⏳ [{patient_id}] A descarregar cabeçalho mestre {sig_master}.hea...")
            descarregar_url(f"{folder_url}{sig_master}.hea", master_hea_local)
            time.sleep(DELAY_ENTRE_DOWNLOADS)
            
        if not os.path.exists(master_hea_local):
            continue
            
        with open(master_hea_local, 'r') as f:
            linhas = f.readlines()
            
        segmentos_teoricos = []
        for linha in linhas:
            parts = linha.split()
            if len(parts) >= 2 and not parts[0].startswith('#'):
                seg_name = parts[0]
                if seg_name != '~' and seg_name != sig_master:
                    segmentos_teoricos.append(seg_name)
                    
        for seg_name in sorted(segmentos_teoricos):
            seg_hea_local = os.path.join(local_dir, f"{seg_name}.hea")
            seg_dat_local = os.path.join(local_dir, f"{seg_name}.dat")
            
            if os.path.exists(seg_dat_local):
                continue
                
            ja_tinha_hea_local = os.path.exists(seg_hea_local)
            if not ja_tinha_hea_local:
                sucesso_header = descarregar_url(f"{folder_url}{seg_name}.hea", seg_hea_local)
                time.sleep(DELAY_ENTRE_DOWNLOADS)
                if not sucesso_header:
                    continue
            
            try:
                with open(seg_hea_local, 'r') as f_seg:
                    conteudo = f_seg.read().upper()
                    
                tem_abp = 'ABP' in conteudo
                tem_pleth = 'PLETH' in conteudo
                tem_ecg = any(x in conteudo for x in ['II', 'V', 'ECG', 'MLII'])
                
                if tem_abp and tem_pleth and tem_ecg:
                    print(f"[{patient_id} | {janelas_atuais} jan] VÁLIDO! A baixar: {seg_name}...", end="", flush=True)
                    
                    descarregar_url(f"{folder_url}{seg_name}.dat", seg_dat_local)
                    time.sleep(DELAY_ENTRE_DOWNLOADS)
                    
                    seg_mat_local = os.path.join(local_dir, f"{seg_name}.mat")
                    descarregar_url(f"{folder_url}{seg_name}.mat", seg_mat_local)
                    time.sleep(DELAY_ENTRE_DOWNLOADS)
                    
                    print(" ")
                    total_novos_segmentos += 1
                else:
                    if not ja_tinha_hea_local and os.path.exists(seg_hea_local):
                        os.remove(seg_hea_local)
            except Exception as e:
                print(f"Erro ao analisar o segmento {seg_name}: {e}")

print("\n" + "="*60)
print("RELATÓRIO FINAL DE HIGIENIZAÇÃO RIGOROSA")
print("="*60)
print(f"Novos segmentos síncronos injetados: {total_novos_segmentos}")
print("="*60)

In [ ]:
import os
import time
import urllib.request
import pandas as pd
from urllib.error import HTTPError

# ==============================================================================
# CONFIGURAÇÕES DE DIRETORIA, CSVs E LIMIAR
# ==============================================================================
PASTA_DATASET = "/media/filipe/Dados/MIMIC_DATASET_CONSOLIDADO"
FICHEIRO_ANCORA = "../../03_ABSOLUTO_codigo_e_resultados/RESULTADOS/pacientes_validos_cohort.txt"

FICHEIROS_CSV = [

    "/media/filipe/Dados/MIMIC_DATASET_CONSOLIDADO/features_mulheres_teste_novo2.csv",

    "/media/filipe/Dados/MIMIC_DATASET_CONSOLIDADO/features_homens_teste_novo2.csv"

]
LIMIAR_JANELAS = 500 

# Base do MIMIC-III Waveform Matched (onde estão os subsegmentos de alta frequência)
BASE_URL = "https://physionet.org/files/mimic3wdb-matched/1.0/"
DELAY_ENTRE_DOWNLOADS = 0.1
# ==============================================================================

def descarregar_url(url, destino_local):
    """Função auxiliar para descarregar ficheiros simulando um browser."""
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as resposta, open(destino_local, 'wb') as f_saida:
            f_saida.write(resposta.read())
        return True
    except HTTPError as e:
        if e.code == 404:
            return False
        print(f"\nErro HTTP {e.code} em: {url}")
        return False
    except Exception as e:
        print(f"\nErro inesperado no download: {e}")
        return False

# --- PASSO 1: Carregar os Doentes Válidos (Âncora do TXT) ---
if not os.path.exists(FICHEIRO_ANCORA):
    raise FileNotFoundError(f"Ficheiro mestre '{FICHEIRO_ANCORA}' não encontrado!")

with open(FICHEIRO_ANCORA, "r") as f:
    pacientes_validos = set(linha.strip() for linha in f if linha.strip())

print(f"Ficheiro Âncora lido: {len(pacientes_validos)} pacientes mapeados no TXT.")

# --- PASSO 2: Carregar e Normalizar os CSVs de Características ---
print("A contabilizar janelas globais dos CSVs (Mulheres + Homens)...")
contagens_lista = []

for csv_path in FICHEIROS_CSV:
    if os.path.exists(csv_path):
        df_tmp = pd.read_csv(csv_path)
        if 'subject_id' in df_tmp.columns:
            df_tmp['subject_id'] = df_tmp['subject_id'].astype(str).str.replace('p', '', case=False)
            contagens_lista.append(df_tmp['subject_id'].value_counts())

contagem_janelas = pd.concat(contagens_lista).groupby(level=0).sum() if contagens_lista else pd.Series()

# --- PASSO 3: Filtrar apenas os doentes críticos dentro da coorte ---
pacientes_criticos = set()
for p in pacientes_validos:
    p_limpo = p.replace('p', '')
    total_janelas_no_csv = contagem_janelas.get(p_limpo, 0)
    if total_janelas_no_csv < LIMIAR_JANELAS:
        pacientes_criticos.add(p)

print(f"Filtragem Concluída: {len(pacientes_criticos)} dos {len(pacientes_validos)} pacientes estão abaixo de {LIMIAR_JANELAS} janelas.")

if not pacientes_criticos:
    print("Incrível! Todos os pacientes do TXT já têm dados suficientes. Processo encerrado.")
    exit()

# --- PASSO 4: Varrer o disco, localizando os caminhos e os headers já descarregados ---
mapeamento_local = {}
print("A varrer o disco para localizar caminhos e ficheiros .hea existentes...")

for raiz, diretorias, ficheiros in os.walk(PASTA_DATASET):
    partes = os.path.normpath(raiz).split(os.sep)
    id_paciente = next((p for p in partes if p in pacientes_criticos), None)
    
    if not id_paciente:
        continue
        
    if id_paciente not in mapeamento_local:
        mapeamento_local[id_paciente] = {'raiz': raiz, 'masters': set()}
        
    for f in ficheiros:
        # Só os cabeçalhos mestres (puramente numéricos) já descarregados
        if f.endswith('.hea'):
            nome_base = f.split('.')[0]
            if nome_base.isdigit():
                mapeamento_local[id_paciente]['masters'].add(nome_base)

# --- PASSO 5: Auditoria sobre os ficheiros locais ---
print("\nA iniciar a auditoria baseada nos cabeçalhos locais...")
total_novos_segmentos = 0

for patient_id in sorted(list(pacientes_criticos)):
    subpasta_p = patient_id[:3]
    p_limpo = patient_id.replace('p', '')
    janelas_atuais = contagem_janelas.get(p_limpo, 0)
    
    print(f"\nDoente: [{patient_id} | {janelas_atuais} jan]", end="", flush=True)
    
    if patient_id in mapeamento_local and mapeamento_local[patient_id]['masters']:
        local_dir = mapeamento_local[patient_id]['raiz']
        masters = mapeamento_local[patient_id]['masters']
    else:
        print(" Ignorado: Sem ficheiros .hea mestres locais para este paciente.", flush=True)
        continue
        
    print(f" -> A ler os ficheiros locais para o(s) Master(s): {sorted(list(masters))}...", flush=True)
    
    baixados_no_paciente = 0
    ja_existiam_no_paciente = 0
    rejeitados_no_paciente = 0
    
    for sig_master in sorted(masters):
        # O URL remoto correto para os segmentos clínicos emparelhados (matched)
        folder_url = f"{BASE_URL}{subpasta_p}/{patient_id}/"
        master_hea_local = os.path.join(local_dir, f"{sig_master}.hea")
        
        # Se faltar algum, salta-se para evitar erros 404
        if not os.path.exists(master_hea_local):
            continue
                
        # Lê o conteúdo do ficheiro mestre local
        with open(master_hea_local, 'r') as f_m:
            linhas = f_m.readlines()
            
        segmentos_teoricos = []
        for linha in linhas:
            parts = linha.split()
            if len(parts) >= 2 and not parts[0].startswith('#'):
                seg_name = parts[0]
                if seg_name != '~' and seg_name != sig_master:
                    segmentos_teoricos.append(seg_name)
                    
        for seg_name in sorted(segmentos_teoricos):
            seg_hea_local = os.path.join(local_dir, f"{seg_name}.hea")
            seg_dat_local = os.path.join(local_dir, f"{seg_name}.dat")
            
            # Se o arquivo de sinal .dat já existe, não faz nada
            if os.path.exists(seg_dat_local):
                ja_existiam_no_paciente += 1
                continue
                
            # Se não tiver o .hea do subsegmento, descarrega do servidor matched
            ja_tinha_hea_local = os.path.exists(seg_hea_local)
            if not ja_tinha_hea_local:
                sucesso_header = descarregar_url(f"{folder_url}{seg_name}.hea", seg_hea_local)
                time.sleep(DELAY_ENTRE_DOWNLOADS)
                if not sucesso_header:
                    continue
            
            try:
                with open(seg_hea_local, 'r') as f_seg:
                    conteudo = f_seg.read().upper()
                    
                # Validação dos canais alvo da extração
                tem_abp = 'ABP' in conteudo
                tem_pleth = 'PLETH' in conteudo
                tem_ecg = any(x in conteudo for x in ['II', 'V', 'ECG', 'MLII'])
                
                if tem_abp and tem_pleth and tem_ecg:
                    print(f"\n   Segmento Alvo VÁLIDO detetado: {seg_name}... A descarregar sinais...", end="", flush=True)
                    
                    # Baixa os binários pesados de sinais (.dat e .mat)
                    descarregar_url(f"{folder_url}{seg_name}.dat", seg_dat_local)
                    time.sleep(DELAY_ENTRE_DOWNLOADS)
                    
                    seg_mat_local = os.path.join(local_dir, f"{seg_name}.mat")
                    descarregar_url(f"{folder_url}{seg_name}.mat", seg_mat_local)
                    time.sleep(DELAY_ENTRE_DOWNLOADS)
                    
                    print(" Concluído!", end="", flush=True)
                    total_novos_segmentos += 1
                    baixados_no_paciente += 1
                else:
                    rejeitados_no_paciente += 1
                    # Se descarregámos o .hea temporariamente e ele não serve, limpa-se o lixo
                    if not ja_tinha_hea_local and os.path.exists(seg_hea_local):
                        os.remove(seg_hea_local)
            except Exception as e:
                print(f"\n   Erro de leitura no segmento {seg_name}: {e}")

    print(f"\n   Resumo: {baixados_no_paciente} novos baixados | {ja_existiam_no_paciente} já guardados | {rejeitados_no_paciente} sem o trio.")

print("\n" + "="*60)
print("RELATÓRIO FINAL DE ATUALIZAÇÃO")
print("="*60)
print(f"Total de novos segmentos síncronos injetados na Cohort: {total_novos_segmentos}")
print("="*60)

In [ ]:
import pandas as pd

# ==============================================================================
# CONFIGURAÇÃO
# ==============================================================================
# Caminho do ficheiro de características consolidado
CAMINHO_CSV =  {
    "/media/filipe/Dados/MIMIC_DATASET_CONSOLIDADO/features_mulheres_teste_novo2.csv" ,
    "/media/filipe/Dados/MIMIC_DATASET_CONSOLIDADO/features_homens_teste_novo2.csv"

}
# ==============================================================================

for caminho in CAMINHO_CSV:

    # 1. Carregar o dataset de features
    print("A carregar o dataset...")
    df = pd.read_csv(caminho)

    # 2. Fazer a contagem de janelas por paciente
    # O value_counts() agrupa por ID e ordena do maior para o menor automaticamente
    contagem_janelas = df['subject_id'].value_counts()

    # 3. Mostrar o resultado
    print("\nCONTAGEM DE JANELAS POR PACIENTE:")
    print("-" * 40)
    print(contagem_janelas)
    print("-" * 40)

    # --- Métricas Extra Úteis ---
    print(f"Total de janelas processadas (Soma): {contagem_janelas.sum()}")
    print(f"Média de janelas por paciente: {contagem_janelas.mean():.1f}")
    print(f"Mínimo de janelas num paciente: {contagem_janelas.min()}")
    print(f"Máximo de janelas num paciente: {contagem_janelas.max()}")

In [ ]:
import os
import time
import urllib.request
import pandas as pd
from urllib.error import HTTPError

# ==============================================================================
# CONFIGURAÇÕES DE DIRETORIA, CSVs E LIMIAR
# ==============================================================================
PASTA_DATASET = "/media/filipe/Dados/MIMIC_DATASET_CONSOLIDADO"
FICHEIRO_ANCORA = "teste.txt"

FICHEIROS_CSV = [
    f"{PASTA_DATASET}/features_mulheres_teste_novo2.csv",
    f"{PASTA_DATASET}/features_homens_teste_novo2.csv"
]

LIMIAR_JANELAS = 100 

# Base Matched do MIMIC-III (onde residem os subsegmentos de alta frequência)
BASE_URL = "https://physionet.org/files/mimic3wdb-matched/1.0/"
DELAY_ENTRE_DOWNLOADS = 0.1

# QUANTOS SEGMENTOS TESTAR POR MASTER (Margem de segurança alargada)
MAX_SUFIXOS_A_TESTAR = 40 
# ==============================================================================

def descarregar_url(url, destino_local):
    """Função auxiliar para descarregar ficheiros simulando um browser."""
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as resposta, open(destino_local, 'wb') as f_saida:
            f_saida.write(resposta.read())
        return True
    except HTTPError as e:
        if e.code == 404:
            return False
        print(f"\nErro HTTP {e.code} em: {url}")
        return False
    except Exception as e:
        print(f"\nErro inesperado no download: {e}")
        return False

# --- PASSO 1: Carregar os Doentes Válidos ---
if not os.path.exists(FICHEIRO_ANCORA):
    raise FileNotFoundError(f"Ficheiro mestre '{FICHEIRO_ANCORA}' não encontrado!")

with open(FICHEIRO_ANCORA, "r") as f:
    pacientes_validos = set(linha.strip() for linha in f if linha.strip())

print(f"Ficheiro Âncora lido: {len(pacientes_validos)} pacientes mapeados no TXT.")

# --- PASSO 2: Carregar Contagem Atual de Janelas ---
print("A contabilizar janelas globais dos CSVs (Mulheres + Homens)...")
contagens_lista = []
for csv_path in FICHEIROS_CSV:
    if os.path.exists(csv_path):
        df_tmp = pd.read_csv(csv_path)
        if 'subject_id' in df_tmp.columns:
            df_tmp['subject_id'] = df_tmp['subject_id'].astype(str).str.replace('p', '', case=False)
            contagens_lista.append(df_tmp['subject_id'].value_counts())

contagem_janelas = pd.concat(contagens_lista).groupby(level=0).sum() if contagens_lista else pd.Series()

# --- PASSO 3: Filtrar Pacientes Necessitados ---
pacientes_criticos = set()
for p in pacientes_validos:
    p_limpo = p.replace('p', '')
    if contagem_janelas.get(p_limpo, 0) < LIMIAR_JANELAS:
        pacientes_criticos.add(p)

print(f"Filtragem Concluída: {len(pacientes_criticos)} pacientes precisam de mais segmentos.")

if not pacientes_criticos:
    print("Todos os pacientes já superaram o limiar.")
    exit()

# --- PASSO 4: Varrer Disco para mapear os IDs Mestres que o paciente possui ---
mapeamento_local = {}
print("A detetar caminhos e IDs Mestres locais...")

for raiz, diretorias, ficheiros in os.walk(PASTA_DATASET):
    partes = os.path.normpath(raiz).split(os.sep)
    id_paciente = next((p for p in partes if p in pacientes_criticos), None)
    
    if not id_paciente:
        continue
        
    if id_paciente not in mapeamento_local:
        mapeamento_local[id_paciente] = {'raiz': raiz, 'masters': set()}
        
    for f in ficheiros:
        if f.endswith('.hea'):
            nome_base = f.split('.')[0]
            # Se encontrar um ficheiro puramente numérico ou o prefixo antes do '_'
            id_mestre_puro = nome_base.split('_')[0]
            if id_mestre_puro.isdigit():
                mapeamento_local[id_paciente]['masters'].add(id_mestre_puro)

# --- PASSO 5: Varredura Cega Remota (Deep Discovery) ---
print("\nA iniciar Varredura Cega Remota por força sequencial...")
total_novos_segmentos = 0

for patient_id in sorted(list(pacientes_criticos)):
    subpasta_p = patient_id[:3]
    p_limpo = patient_id.replace('p', '')
    janelas_atuais = contagem_janelas.get(p_limpo, 0)
    
    if patient_id not in mapeamento_local or not mapeamento_local[patient_id]['masters']:
        print(f"\nDoente: [{patient_id} | {janelas_atuais} jan] Sem IDs Mestres locais para rastrear.")
        continue
        
    local_dir = mapeamento_local[patient_id]['raiz']
    masters = mapeamento_local[patient_id]['masters']
    
    print(f"\nDoente: [{patient_id} | {janelas_atuais} jan] -> Forçando varrimento de {len(masters)} blocos de exames...")
    
    baixados_no_paciente = 0
    rejeitados_no_paciente = 0
    
    folder_url = f"{BASE_URL}{subpasta_p}/{patient_id}/"
    
    for sig_master in sorted(masters):
        print(f"   Rastreando caminhos para o Mestre {sig_master}: ", end="", flush=True)
        
        # Tentamos descobrir segmentos de _0001 até _0040 de forma cega
        for i in range(1, MAX_SUFIXOS_A_TESTAR + 1):
            seg_name = f"{sig_master}_{i:04d}"  # Gera 3271290_0001, 3271290_0002...
            
            seg_hea_local = os.path.join(local_dir, f"{seg_name}.hea")
            seg_dat_local = os.path.join(local_dir, f"{seg_name}.dat")
            seg_mat_local = os.path.join(local_dir, f"{seg_name}.mat")
            
            # Se o .dat já existe localmente, salta-se
            if os.path.exists(seg_dat_local):
                continue
                
            # Tenta descarregar o .hea temporariamente de forma cega do servidor
            ja_tinha_hea = os.path.exists(seg_hea_local)
            if not ja_tinha_hea:
                sucesso_hea = descarregar_url(f"{folder_url}{seg_name}.hea", seg_hea_local)
                time.sleep(DELAY_ENTRE_DOWNLOADS)
                if not sucesso_hea:
                    # Se falhar o _0001 ou o _0015 e der 404, não quebra imediatamente, 
                    # porque o MIMIC pode ter saltos na numeração. Continuamos o loop.
                    continue
            
            # Se o .hea existe ou foi baixado com sucesso, validamos o conteúdo por canais
            try:
                with open(seg_hea_local, 'r') as f_seg:
                    conteudo = f_seg.read().upper()
                
                tem_abp = 'ABP' in conteudo
                tem_pleth = 'PLETH' in conteudo
                tem_ecg = any(x in conteudo for x in ['II', 'V', 'ECG', 'MLII'])
                
                if tem_abp and tem_pleth and tem_ecg:
                    print(f"[{seg_name} ] ", end="", flush=True)
                    
                    # Descarrega os binários pesados de sinais em falta
                    descarregar_url(f"{folder_url}{seg_name}.dat", seg_dat_local)
                    time.sleep(DELAY_ENTRE_DOWNLOADS)
                    descarregar_url(f"{folder_url}{seg_name}.mat", seg_mat_local)
                    time.sleep(DELAY_ENTRE_DOWNLOADS)
                    
                    total_novos_segmentos += 1
                    baixados_no_paciente += 1
                else:
                    rejeitados_no_paciente += 1
                    # Elimina o .hea temporário se ele não tiver o trio síncrono
                    if not ja_tinha_hea and os.path.exists(seg_hea_local):
                        os.remove(seg_hea_local)
                        
            except Exception as e:
                print(f"\n   Erro ao processar teste do segmento {seg_name}: {e}")
                
        print("Fim do bloco.")
    print(f"   Resultado do Paciente: {baixados_no_paciente} novos injetados | {rejeitados_no_paciente} descartados por falta de canais.")

print("\n" + "="*60)
print("FIM DA VARREDURA CEGA EXAUSTIVA")
print("="*60)
print(f"Total absoluto de novos segmentos descobertos e baixados: {total_novos_segmentos}")
print("="*60)